> ### 🔑 Before you run anything: add your OpenRouter key as a Colab secret
>
> Every LLM call in this notebook (the agent **and** the scorer's free-text judge)
> goes through **OpenRouter** (`https://openrouter.ai`).
>
> 1. Create a key at https://openrouter.ai/keys (it starts with `sk-or-v1-`).
> 2. In Colab click the key icon (🔑) in the left sidebar → **Add new secret**:
>    name `OPENROUTER_API_KEY`, value = your key, and switch on **Notebook access**.
> 3. That's it — no GPU runtime is needed; the default CPU runtime is fine.
>
> Never paste the key into a cell: this notebook is tracked in git.

# 🤖 AgenTekki — Tax Agent Filler

### 🏗️ What you will build

You are going to build an AI agent that autonomously fills out a Swiss tax return  end to end without hardcoded answers. The agent reads documents, reasons about Swiss tax law, navigates a real web form, and submits a completed declaration.

Along the way you will get hands-on experience with two of the most important patterns in modern AI engineering:

- 🔧 **Tool use**  giving an LLM the ability to interact with the world: read files, click buttons, fill fields, submit forms.
- 🧠 **Step-by-step reasoning**  watching a model think through a problem one action at a time, building on what it already knows.

### ⏱️ Time budget

Plan for **about 2–3 hours**. Everything runs on the free CPU runtime. A full agent run takes a
few minutes per persona and a few cents of OpenRouter credit. Nothing here should run for
hours — if a cell does, stop it and go back to the last ✅ checkpoint.

### 🧪 What is real here, and what is a teaching stand-in

| Piece | In this notebook | In practice |
|---|---|---|
| The tax form | A React **simulator** of Zurich's ZHprivateTax portal (`EnvDemoV1/`) | The real cantonal e-filing portal |
| The taxpayer's documents | **Synthetic** personas in `personas/` — Lohnausweis, bank CSV, receipts as plain text | PDFs and scans uploaded by a real client |
| The taxpayer | A **simulated** person: `ask_user` answers from a scripted knowledge file | A human you would email or call |
| The tax rules | **Simplified excerpts** in `guides/` (`policy_kb.txt` + topical guides) | The official Wegleitung and Steuerbuch |
| The LLM | Real — any model on OpenRouter | The same |

Happy coding 🚀

## 🎓 Before you start

### 📚 What you already know

We assume you are comfortable with:
- Python functions, dictionaries, lists and **f-strings** ([refresher](https://docs.python.org/3/tutorial/inputoutput.html#formatted-string-literals)); classes and methods appear but you only *call* them ([primer](https://docs.python.org/3/tutorial/classes.html))
- JSON as a data format (`json.dumps` / `json.loads`)
- The idea of calling a chat LLM with a **system prompt** and a **user prompt** — Section 6 recaps this in one table

No tax knowledge is needed: the rules the agent must apply are in the guides it reads.

### 🎯 Learning outcomes

By the end of this notebook you will:
- Understand the **Plan-Execute** pattern in AI agents
- Design **system vs. user prompts** for two different jobs done by the same model
- Implement **selective document retrieval** (the LLM chooses what to read)
- **Visualize** agent decision-making (plans, document selection, reasoning)
- Wire a **full autonomous pipeline**: scan → plan → retrieve → fill → navigate → submit
- Read a **field-level score card** and reason about *why* a field was right or wrong

### 🗺️ Your tasks at a glance

Every block you must fill is marked with a `# 🎯 TODO` comment. A collapsed **✅ Checkpoint**
cell follows each task — run it before moving on.

| # | Task | Where |
|---|---|---|
| 1 | Write the **income plan** by hand | Section 5 |
| 2 | Build the **plan-phase user message** | Section 6 |
| 3 | Build the **execute-phase user message** | Section 6 |
| 4 | Implement **`think_step`** (retrieve the requested documents, run the execute call) | Section 6 |
| 5 | **Test** `think_step` on one page | Section 6 |
| 6 | Wire `think_step` into **`think_loop`** and run the full agent | Section 8 |


## 1. ⚙️ Setup

In [1]:
#@title Clone Repository & Install Dependencies (click to run) { display-mode: "form" }

import os, sys, subprocess

GITHUB_REPO   = "github.com/eth-bmai-hs26/w2-project-public.git"
GITHUB_BRANCH = "main"
REPO_DIR      = "/content/w2-project-public"

# The course repo is public — this clones anonymously, no token needed.

os.chdir("/content")

!rm -rf /content/w2-project-public

_clone = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", GITHUB_BRANCH,
     f"https://{GITHUB_REPO}", REPO_DIR],
    capture_output=True, text=True,
)
if _clone.returncode != 0:
    raise RuntimeError(
        "git clone failed.\n\n" + _clone.stderr.strip() + "\n\n"
        "Check your Colab runtime has network access, then re-run this cell."
    )
del _clone

# Purge cached modules if re-running
for mod_name in list(sys.modules):
    if "mcp_server" in mod_name:
        del sys.modules[mod_name]

sys.path.insert(0, REPO_DIR)

!pip install -q openai

print("Done!")


Done!


In [ ]:
#@title Build & Serve the React Frontend (click to run) { display-mode: "form" }

import subprocess, os, threading, http.server, functools, socket

FRONTEND_DIR = "/content/w2-project-public"
BUILD_DIR = os.path.join(FRONTEND_DIR, "EnvDemoV1", "dist")

# Install and build
os.chdir(os.path.join(FRONTEND_DIR, "EnvDemoV1"))
subprocess.run(["npm", "install"], check=True)
subprocess.run(["npm", "run", "build"], check=True)
os.chdir("/content")

# Find a free port
def _find_free_port(start=3000, end=3020):
    for port in range(start, end):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("", port))
                return port
            except OSError:
                continue
    raise RuntimeError("No free port found")

PORT = _find_free_port()
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory=BUILD_DIR)
httpd = http.server.HTTPServer(("", PORT), handler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"Frontend served at http://localhost:{PORT}")

### 🎯 Set up your OpenRouter API key

Add the `OPENROUTER_API_KEY` secret (🔑 icon in the left sidebar) before running the next cell.

In [ ]:
#@title Initialize the LLM service — OpenRouter is the only route { display-mode: "form" }
from mcp_server.llm_service import LLMService
from google.colab import userdata

# ══════════════════════════════════════════════════════════════════
# OpenRouter — one key, many models
# ══════════════════════════════════════════════════════════════════
# OpenRouter speaks the OpenAI API, so it plugs straight into
# LLMService.openai_compatible — no extra dependency needed.
#
#   1. Create a key at https://openrouter.ai/keys  (starts with sk-or-v1-)
#   2. In Colab, click the key icon (🔑) in the left sidebar,
#      then "Add new secret":
#        Name  : OPENROUTER_API_KEY
#        Value : <paste the key>
#      and turn on "Notebook access" for this notebook.
#
# If the secret is missing you are prompted for the key instead; the
# prompt is hidden and nothing is stored.
#
# Do NOT paste the key into this cell. This notebook is tracked in
# git, so anything typed here is saved into the .ipynb and committed.
#
# Models are named "vendor/model" — browse https://openrouter.ai/models.
# Ids ending in ":free" cost nothing but are rate-limited.
# ══════════════════════════════════════════════════════════════════
# Optional: pick another model at https://openrouter.ai/models
OPENROUTER_MODEL = "google/gemini-3.8-flash"

def _get_secret(name, prompt):
    """Colab secret first, then a hidden prompt. Never written into the notebook."""
    try:
        value = userdata.get(name)
        if value and value.strip():
            return value.strip()
    except Exception:
        pass   # secret missing, notebook access off, or not running in Colab
    import getpass
    return getpass.getpass(prompt).strip()

def create_llm(model=None):
    """OpenRouter-backed LLMService. The key comes from the OPENROUTER_API_KEY secret."""
    return LLMService.openai_compatible(
        model=model or OPENROUTER_MODEL,
        base_url="https://openrouter.ai/api/v1",
        api_key=_get_secret("OPENROUTER_API_KEY", "OpenRouter API key: "),
        default_headers={
            "HTTP-Referer": "https://github.com/eth-bmai-hs26/w2-project-public",
            "X-Title": "AgenTekki Tax Filler",
        },
    )

llm = create_llm()

# quick test. Thinking models (Gemini, o-series, ...) spend part of the reply
# budget on reasoning before they answer, so give the one-word answer some room.
response = llm.chat(
    system="You are a helpful assistant.",
    user="Say 'hello' in German. Reply with just the word.",
    max_tokens=300,
)
if response.strip():
    print(f"LLM says: {response.strip()}")
else:
    print("LLM says: (nothing) — the model used the whole reply budget thinking. "
          "Run the cell again, or raise max_tokens above.")


## 2. 🖥️ Simulation — Explore the Tax Portal

Before building an agent, understand what we're automating. The cell below renders the tax portal UI. Click around and explore!

**Try this:**
- Navigate through the sections using the sidebar (Personal, Income, Deductions, Wealth)
- Click "Next" to move between sub-pages
- Add rows in the Children or Bank Accounts sections
- Check the Review page to see how all data comes together
- Notice the field types: text inputs, number inputs, dropdowns, checkboxes

In [ ]:
#@title 🎯 Explore the Tax Portal (click around, nothing is saved!) { display-mode: "form" }

from IPython.display import display, HTML
import os

BUILD_DIR = "/content/w2-project-public/EnvDemoV1/dist"
with open(os.path.join(BUILD_DIR, "index.html")) as f:
    html_content = f.read()

html_content = html_content.replace(r'"\/assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"./assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"/assets/', f'"http://localhost:{PORT}/assets/')

display(HTML(html_content))

**🎯 Reflection** (discuss with your team):
- How many pages does the form have? How many sub-pages?
- What kinds of fields appear? (text, number, dropdown, checkbox, date)
- If you were an AI agent, which documents would you need for the Income page? For Deductions?
- What information can you get from the field *labels* alone, without reading any documents?

## 3. 📂 Available Documents & Guides

Before building plans, let's see what documents and tax guides the agent has access to.

In [ ]:
#@title Initialize the MCP server with MockBridge (no real browser yet) { display-mode: "form" }
from mcp_server.server import MCPServer
from mcp_server.bridges.mock import MockBridge
from mcp_server.doc_catalog import build_document_catalog

REPO = "/content/w2-project-public"
PERSONA = "anna_meier"

mock_server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=MockBridge(),
)

# Build the document catalog
catalog = build_document_catalog(mock_server)
print(f"Loaded {len(catalog)} documents for {PERSONA}")

In [ ]:
#@title Display document catalog { display-mode: "form" }
from IPython.display import display, HTML

# Colors follow Colab's theme variables so the output is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
TABLE_STYLE = """
<style>
  .nb-table { color: var(--colab-primary-text-color, inherit); font-family: sans-serif; }
  .nb-table table { border-collapse: collapse; width: 100%; }
  .nb-table th { padding: 8px; text-align: left; color: #fff; }
  .nb-table th.blue { background: #4a90d9; }
  .nb-table th.green { background: #2d7d2d; }
  .nb-table td {
    padding: 6px 8px; vertical-align: top;
    border-bottom: 1px solid var(--colab-border-color, rgba(128,128,128,.35));
  }
  .nb-table code {
    color: inherit;
    background: var(--colab-highlighted-surface-color, rgba(128,128,128,.15));
    padding: 1px 4px; border-radius: 3px;
  }
  .nb-table .muted { color: var(--colab-secondary-text-color, #888); }
</style>
"""

rows = "".join(
    f"<tr><td><code>{d['filename']}</code></td><td>{d['description']}</td></tr>"
    for d in catalog
)
display(HTML(TABLE_STYLE + f'''
<div class="nb-table">
<h3>Documents available for {PERSONA}</h3>
<table>
<tr><th class="blue">Filename</th><th class="blue">Description</th></tr>
{rows}
</table>
<p><b>Note:</b> The Plan phase sees only this table — not the actual file contents.</p>
</div>
'''))

In [ ]:
#@title Fetch and display tax guides { display-mode: "form" }
from IPython.display import display, HTML
import json
from pathlib import Path

guides = mock_server.list_guides()

# Load catalog descriptions
catalog_path = Path(f"{REPO}/guides/guide_catalog.json")
guide_descs = {}
if catalog_path.exists():
    with open(catalog_path) as f:
        raw_catalog = json.load(f)
    guide_descs = {k: v["description"] for k, v in raw_catalog.items()}

print(f"Found {len(guides)} guides")

# Colors follow Colab's theme variables so the table is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
STYLE = """
<style>
  .guides { color: var(--colab-primary-text-color, inherit); }
  .guides table { border-collapse: collapse; width: 100%; }
  .guides th {
    padding: 8px; text-align: left;
    background: #2d7d2d; color: #fff;
  }
  .guides td {
    padding: 6px 8px; vertical-align: top;
    border-bottom: 1px solid var(--colab-border-color, rgba(128,128,128,.35));
  }
  .guides td.desc { font-size: 13px; }
  .guides code {
    color: inherit;
    background: var(--colab-highlighted-surface-color, rgba(128,128,128,.15));
    padding: 1px 4px; border-radius: 3px;
  }
  .guides .muted {
    color: var(--colab-secondary-text-color, #888); font-style: italic;
  }
</style>
"""

if guides:
    rows = ""
    for g in guides:
        desc = guide_descs.get(g["topic"], "<span class='muted'>no description</span>")
        rows += (
            f"<tr>"
            f"<td><code>{g['title']}</code></td>"
            f"<td><code>{g['topic']}</code></td>"
            f"<td class='desc'>{desc}</td>"
            f"</tr>"
        )
    display(HTML(STYLE + f'''
    <div class="guides">
    <h3>Tax Guides</h3>
    <table>
    <tr>
        <th>Title</th>
        <th>Topic</th>
        <th>Description</th>
    </tr>
    {rows}
    </table>
    <p><b>Note:</b> This table is the catalog only. The next cell shows how to open a guide and read its full text.</p>
    </div>
    '''))
else:
    display(HTML(STYLE + '''
    <div class="guides">
    <h3>Tax Guides</h3>
    <p class="muted">No guides found.</p>
    </div>
    '''))

### 📖 Read a guide yourself

The table above is all the **Plan phase** gets to see: a title, a topic and one sentence.
The rules themselves (thresholds, flat rates, caps) live in the full text, which only
reaches the model in the **Execute phase**, and only for the guides the plan asked for.

Two tool calls get you there, exactly the way the agent does it:

1. `mock_server.list_guides()` returns one entry per guide with a `topic` and a `url` (the file path).
2. `mock_server.fetch_guide(url)` returns `{"title", "content", ...}` — the raw file. Ten of the
   eleven guides are HTML blog articles, so the raw content starts with a `<style>` block; strip
   the tags before reading it (and before sending it to a model).

Run the cell, read the opening of the first guide, then open any other guide from the
drop-downs below the output.

In [ ]:
#@title Read a guide the way the agent does { display-mode: "form" }
import re, html
from IPython.display import display, HTML

def html_to_text(raw):
    """Turn a guide file into readable plain text: drop <style>/<script>, strip tags, tidy whitespace."""
    text = re.sub(r"<(style|script)[^>]*>.*?</\1>", " ", raw, flags=re.S | re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("&nbsp;", " ").replace("&amp;", "&")
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n\s*\n+", "\n", text).strip()

def read_guide(topic):
    """Fetch one guide by its topic (the Topic column above) and return its plain text."""
    entry = next(g for g in mock_server.list_guides() if g["topic"] == topic)
    return html_to_text(mock_server.fetch_guide(entry["url"])["content"])

guides = mock_server.list_guides()

# 1. Open the first guide in the catalog — the same two calls the agent would make.
first = guides[0]
raw = mock_server.fetch_guide(first["url"])      # fetch_guide wants the url, not the topic
text = html_to_text(raw["content"])

print(f"Guide: {first['title']}  (topic: {first['topic']})")
print(f"Raw file: {len(raw['content'])} characters of HTML -> {len(text)} characters of text\n")
print("--- The opening of the guide ---")
print(text[:700] + (" [...]" if len(text) > 700 else ""))

# 2. Every guide in full, one drop-down each.
# Colors follow Colab's theme variables so the list is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
GUIDE_STYLE = """
<style>
  .guide-list { color: var(--colab-primary-text-color, inherit); font-family: sans-serif; margin-top: 14px; }
  .guide-list details {
    border: 1px solid var(--colab-border-color, rgba(128,128,128,.35));
    border-radius: 6px; padding: 6px 10px; margin: 6px 0;
  }
  .guide-list summary { cursor: pointer; font-weight: bold; }
  .guide-list summary code {
    color: var(--colab-secondary-text-color, #888); font-weight: normal; margin-left: 6px;
  }
  .guide-list pre { white-space: pre-wrap; font-size: 13px; margin: 8px 0 4px 0; }
</style>
"""
blocks = "".join(
    f"<details><summary>{g['title']}<code>{g['topic']}</code></summary>"
    f"<pre>{html.escape(read_guide(g['topic']))}</pre></details>"
    for g in guides
)
display(HTML(GUIDE_STYLE + f'<div class="guide-list"><b>Full text of every guide</b> (click to expand){blocks}</div>'))

## 4. 🧰 Available Tools

Let's see the tools the agent can use to interact with the tax portal.

In [ ]:
#@title Display available tools { display-mode: "form" }
from IPython.display import display, HTML

# Colors follow Colab's theme variables so the output is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
TABLE_STYLE = """
<style>
  .nb-table { color: var(--colab-primary-text-color, inherit); font-family: sans-serif; }
  .nb-table table { border-collapse: collapse; width: 100%; }
  .nb-table th { padding: 8px; text-align: left; color: #fff; }
  .nb-table th.blue { background: #4a90d9; }
  .nb-table th.green { background: #2d7d2d; }
  .nb-table td {
    padding: 6px 8px; vertical-align: top;
    border-bottom: 1px solid var(--colab-border-color, rgba(128,128,128,.35));
  }
  .nb-table code {
    color: inherit;
    background: var(--colab-highlighted-surface-color, rgba(128,128,128,.15));
    padding: 1px 4px; border-radius: 3px;
  }
  .nb-table .muted { color: var(--colab-secondary-text-color, #888); }
</style>
"""

tools_info = [
    ("scan_page()", "Returns all visible fields, buttons, and text on the current form page"),
    ("fill_field(locator, value)", "Enters a value into a form field identified by its locator"),
    ("click_element(locator)", "Clicks a button or navigation link"),
    ("submit_form()", "Submits the completed tax return"),
    ("list_documents()", "Lists all files in the persona's document folder"),
    ("read_document(filepath)", "Reads and extracts content from a document (.txt, .csv, .pdf, .json)"),
    ("list_guides()", "Returns available tax guide topics"),
    ("fetch_guide(url)", "Returns the full text of a specific tax guide"),
    ("ask_user(question)", "Asks the simulated taxpayer a question (penalized if unnecessary)"),
]

rows = "".join(
    f"<tr><td><code>server.{name}</code></td><td>{desc}</td></tr>"
    for name, desc in tools_info
)
display(HTML(TABLE_STYLE + f'''
<div class="nb-table">
<h3>Available Tools</h3>
<table>
<tr><th class="green">Tool</th><th class="green">Description</th></tr>
{rows}
</table>
</div>
'''))

In [ ]:
#@title What does scan_page() actually return? { display-mode: "form" }
from mcp_server.bridges.mock import MockBridge

mock = MockBridge()
mock.click_element("btn-nav-income")   # navigate with the same tool the agent uses
page_data = mock.scan_page()

import json
fillable = [el for el in page_data["elements"] if el.get("type") not in ("button", "text")]
print(f"Page: {page_data['page_name']} — {len(fillable)} fillable fields")

In [ ]:
#@title Display scan_page() results { display-mode: "form" }
from IPython.display import display, HTML

# Colors follow Colab's theme variables so the output is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
TABLE_STYLE = """
<style>
  .nb-table { color: var(--colab-primary-text-color, inherit); font-family: sans-serif; }
  .nb-table table { border-collapse: collapse; width: 100%; }
  .nb-table th { padding: 8px; text-align: left; color: #fff; }
  .nb-table th.blue { background: #4a90d9; }
  .nb-table th.green { background: #2d7d2d; }
  .nb-table td {
    padding: 6px 8px; vertical-align: top;
    border-bottom: 1px solid var(--colab-border-color, rgba(128,128,128,.35));
  }
  .nb-table code {
    color: inherit;
    background: var(--colab-highlighted-surface-color, rgba(128,128,128,.15));
    padding: 1px 4px; border-radius: 3px;
  }
  .nb-table .muted { color: var(--colab-secondary-text-color, #888); }
</style>
"""

rows = "".join(
    f'<tr><td><code>{el.get("type","")}</code></td>'
    f'<td><code>{el.get("locator","")}</code></td>'
    f'<td>{el.get("label","")}</td></tr>'
    for el in fillable
)

display(HTML(TABLE_STYLE + f'''
<div class="nb-table">
<h3>Page: {page_data["page_name"]} &mdash; {len(fillable)} fillable fields</h3>
<table>
<tr><th class="blue">Type</th><th class="blue">Locator</th><th class="blue">Label</th></tr>
{rows}
</table>
<p class="muted" style="margin-top:8px;">This is what the LLM sees when deciding which fields to fill.</p>
</div>
'''))

## 5. 🗺️ Plan Discovery — Think Before You Act

Now that you know which documents are available, let's think about what steps are needed to fill one page of the tax form.

The agent follows a cycle: **Perceive** (scan the page) → **Plan** (decide what docs to read and how to fill) → **Act** (read docs, fill fields).

Below, you'll write plans *by hand* — the same kind of plan the LLM will later generate automatically. Use the document catalog (Section 3) to decide which files you need and the `scan_page()` table (Section 4) for the field locators.

### 🎯 Task 1 — Write the income plan by hand

In [ ]:
# Here's what a plan for the Personal page looks like:
personal_plan = {
    "_reasoning": "Personal info (name, address, DOB, AHV, marital status) comes from profile.json. "
                  "Profession and employer come from the Lohnausweis header.",
    "documents_needed": ["profile.json", "lohnausweis.txt"],
    "guides_needed": [],
    "strategy": "Copy name, address, DOB, AHV and marital status from profile.json. "
                "Take profession and employer from the Lohnausweis header.",
    "fields_to_fill": ["field-personal-main-name", "field-personal-main-ahv",
                       "field-personal-main-address", "field-personal-main-dob",
                       "field-personal-main-zivilstand", "field-personal-main-beruf",
                       "field-personal-main-arbeitgeber"],
    "ask_user_needed": False,
}

# ============================================================
# 🎯 TODO: Task 1 — write the plan for the INCOME page.
# Use the document catalog (Section 3) and the income locators from
# the scan_page() table (Section 4). Which documents does the agent
# need, and which field gets its value from where?
# Hint: the income fields are gross salary, AHV/IV/EO, BVG and
#       interest income. Where does each of them come from?
# ============================================================
income_plan = {
    "_reasoning": "...",        # 🎯 TODO: one or two sentences: which document holds what
    "documents_needed": [],     # 🎯 TODO: file names exactly as they appear in the catalog
    "guides_needed": [],
    "strategy": "...",          # 🎯 TODO: which Lohnausweis field / bank-statement rows feed which locator
    "fields_to_fill": [],       # 🎯 TODO: locators from the scan_page() table
    "ask_user_needed": False,
}

deductions_plan = {
    "_reasoning": "Meal deductions depend on Lohnausweis field 14.1 (canteen subsidy). "
                  "Commuting costs come from the ZVV receipt. Pillar 3a comes from the "
                  "pillar 3a certificate. Donations show up as 'Spende' rows in the bank statement. "
                  "The flat-rate professional-expense deduction is a RULE, not a document value, "
                  "so the policy guide is needed as well.",
    "documents_needed": ["lohnausweis.txt", "bank_statement.csv", "zvv_receipt.txt", "pillar_3a.txt"],
    "guides_needed": ["policy_kb", "velo-pauschale-fahrkosten"],
    "strategy": "Read field 14.1 for the meal allowance and field 12 (Nettolohn) for the "
                "flat-rate professional expenses, then apply the rules in policy_kb. "
                "Use the ZVV receipt for commuting costs and check the Velo-Pauschale guide. "
                "Take the pillar 3a amount from the certificate; sum the 'Spende' rows for donations.",
    "fields_to_fill": ["field-deductions-verpflegung-amount", "field-deductions-flatrate-amount",
                       "field-deductions-fahrkosten-amount", "field-deductions-pillar3a-amount",
                       "field-deductions-donations-amount"],
    "ask_user_needed": False,
}

print("Plans defined for: personal, income, deductions")

In [ ]:
#@title Display plans as HTML cards { display-mode: "form" }
from IPython.display import display, HTML

# Colors follow Colab's theme variables so the cards are readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
CARD_STYLE = """
<style>
  .plan-card {
    border: 2px solid #4a90d9; border-radius: 8px;
    padding: 14px; margin: 10px 0; font-family: sans-serif;
    color: var(--colab-primary-text-color, inherit);
    background: var(--colab-highlighted-surface-color, rgba(74,144,217,.08));
  }
  .plan-card h4 { color: #4a90d9; margin: 0 0 8px 0; }
  .plan-card div { margin: 4px 0; }
  .plan-card code {
    color: inherit;
    background: rgba(128,128,128,.15);
    padding: 1px 4px; border-radius: 3px;
  }
  .plan-card .muted {
    color: var(--colab-secondary-text-color, #888); font-style: italic;
  }
</style>
"""

def _codes(items, empty):
    return ", ".join(f"<code>{x}</code>" for x in items) if items else f"<span class='muted'>{empty}</span>"

display(HTML(CARD_STYLE))
for page_label, plan in [("PERSONAL", personal_plan), ("INCOME", income_plan), ("DEDUCTIONS", deductions_plan)]:
    docs_html = _codes(plan.get("documents_needed", []), "none")
    guides_html = _codes(plan.get("guides_needed", []), "none")
    fields_html = _codes(plan.get("fields_to_fill", []), "not specified")
    reasoning = plan.get("_reasoning", "...")
    strategy = plan.get("strategy", "...")
    display(HTML(f'''
    <div class="plan-card">
        <h4>{page_label}</h4>
        <div><b>Reasoning:</b> {reasoning}</div>
        <div><b>Documents:</b> {docs_html}</div>
        <div><b>Guides:</b> {guides_html}</div>
        <div><b>Strategy:</b> {strategy}</div>
        <div><b>Fields to fill:</b> {fields_html}</div>
    </div>
    '''))

In [ ]:
#@title ✅ Checkpoint — check your income plan { display-mode: "form" }
# Hard-coded expectations for the Anna Meier income page. No LLM call.
_problems = []
_docs = income_plan.get("documents_needed", [])
if not _docs or any(not isinstance(d, str) for d in _docs):
    _problems.append("documents_needed must be a list of file names (strings) exactly as shown in the catalog")
if "lohnausweis.txt" not in _docs:
    _problems.append("the salary fields (gross salary, AHV, BVG) come from lohnausweis.txt — add it")
if "bank_statement.csv" not in _docs:
    _problems.append("interest income comes from bank_statement.csv — add it")
_fields = income_plan.get("fields_to_fill", [])
if not _fields:
    _problems.append("fields_to_fill is empty — copy the income locators from the scan_page() table")
elif not all(str(f).startswith("field-income-") for f in _fields):
    _problems.append("every locator on this page starts with 'field-income-' — check the scan_page() table")
for key in ("_reasoning", "strategy"):
    if "..." in str(income_plan.get(key, "...")):
        _problems.append(f"{key} still contains the '...' placeholder")

if _problems:
    print("❌ Not quite yet:")
    for p in _problems:
        print("   •", p)
else:
    print("✅ income_plan looks complete — the Plan phase of the agent will produce plans like this automatically.")


**Key insight:** You just did exactly what the agent's **Plan phase** will do! The difference: the LLM will write these plans automatically, seeing only the document catalog (filenames and descriptions) — not the actual file contents. This is how the agent decides what information to retrieve before making expensive document reads.

## 6. 🧠 Think Step — Single-Page Reasoning

Now we implement the **two-phase reasoning** for a single page.

### 💬 How LLM calls work: System vs. User prompts

Every call to an LLM has (at least) two parts:

| Role | What it does | Analogy |
|------|-------------|---------|
| **System prompt** | Sets the LLM's persona, rules, and output format. The model treats this as persistent instructions. | A job description pinned to the wall |
| **User prompt** | The actual question or task for *this specific call*. Changes every time. | A new task handed to the employee |

```python
response = llm.chat(
    system="You are an expert Swiss tax accountant. Always reply in JSON.",  # <-- system prompt
    user="Here are the income page fields: ... Fill them using this document: ...",  # <-- user prompt
)
```

**Why does this matter here?** Our agent makes **two LLM calls per page**, each with a *different* system prompt:

1. **Plan call** — system prompt says: *"You are a planning agent. Pick which documents to read. Do NOT fill fields yet."*
2. **Execute call** — system prompt says: *"You are a tax accountant. Given these documents, output `{locator: value}` JSON."*

The system prompt controls *what kind of task* the LLM performs. Same model, different behavior.


In [ ]:
#@title Think Step Architecture Diagram { display-mode: "form" }
from IPython.display import display, HTML

display(HTML('''
<div style="font-family: sans-serif; max-width: 620px; margin: 16px auto;">

  <!-- scan_page -->
  <div style="text-align:center; margin-bottom:4px;">
    <div style="display:inline-block; background:#e3f2fd; border:2px solid #1976d2;
                border-radius:24px; padding:8px 24px; font-weight:bold; color:#1976d2;">
      scan_page()
    </div>
  </div>
  <div style="text-align:center; color:#1976d2; font-size:22px; line-height:1;">&#9660;</div>
  <div style="text-align:center; color:#666; font-size:13px; margin-bottom:8px;">page fields</div>

  <!-- PLAN PHASE -->
  <div style="background:linear-gradient(135deg,#e8f5e9,#c8e6c9); border:2px solid #388e3c;
              border-radius:10px; padding:14px 18px; margin:0 auto 4px auto; max-width:480px;
              position:relative;">
    <div style="font-weight:bold; color:#2e7d32; font-size:15px; margin-bottom:6px;">
      1 &mdash; PLAN PHASE
    </div>
    <div style="background:#f1f8e9; border:1px dashed #66bb6a; border-radius:6px;
                padding:6px 10px; margin-bottom:8px; font-size:12px; color:#555;">
      <b>System prompt:</b> <i>&ldquo;You are a planning agent. Pick which documents to read. Do NOT fill fields yet.&rdquo;</i>
    </div>
    <div style="display:flex; justify-content:space-between; align-items:center;">
      <div style="color:#333; font-size:13px;">
        fields + doc catalog<br/>
        <span style="color:#888; font-size:12px;">(no doc content!)</span>
      </div>
      <div style="color:#2e7d32; font-size:20px; padding:0 12px;">&#10132;</div>
      <div style="background:white; border:1px solid #a5d6a7; border-radius:6px;
                  padding:6px 12px; font-size:13px; color:#333;">
        plan JSON<br/>
        <span style="color:#888; font-size:11px;">docs to read, strategy, reasoning</span>
      </div>
    </div>
  </div>
  <div style="text-align:center; color:#388e3c; font-size:22px; line-height:1;">&#9660;</div>

  <!-- SELECTIVE RETRIEVAL -->
  <div style="background:linear-gradient(135deg,#fff3e0,#ffe0b2); border:2px solid #f57c00;
              border-radius:10px; padding:14px 18px; margin:4px auto; max-width:480px;">
    <div style="font-weight:bold; color:#e65100; font-size:15px; margin-bottom:6px;">
      2 &mdash; SELECTIVE RETRIEVAL
    </div>
    <div style="display:flex; justify-content:space-between; align-items:center;">
      <div style="color:#333; font-size:13px;">
        read only the docs<br/>the plan requested
      </div>
      <div style="color:#e65100; font-size:20px; padding:0 12px;">&#10132;</div>
      <div style="background:white; border:1px solid #ffcc80; border-radius:6px;
                  padding:6px 12px; font-size:13px; color:#333;">
        doc contents
      </div>
    </div>
    <div style="margin-top:8px; font-size:12px; color:#888; font-style:italic;">
      No LLM call &mdash; just filesystem reads based on the plan
    </div>
  </div>
  <div style="text-align:center; color:#f57c00; font-size:22px; line-height:1;">&#9660;</div>

  <!-- EXECUTE PHASE -->
  <div style="background:linear-gradient(135deg,#ede7f6,#d1c4e9); border:2px solid #5e35b1;
              border-radius:10px; padding:14px 18px; margin:4px auto; max-width:480px;">
    <div style="font-weight:bold; color:#4527a0; font-size:15px; margin-bottom:6px;">
      3 &mdash; EXECUTE PHASE
    </div>
    <div style="background:#ede7f6; border:1px dashed #9575cd; border-radius:6px;
                padding:6px 10px; margin-bottom:8px; font-size:12px; color:#555;">
      <b>System prompt:</b> <i>&ldquo;You are an expert Swiss tax accountant. Given these documents, output {locator: value} JSON.&rdquo;</i>
    </div>
    <div style="display:flex; justify-content:space-between; align-items:center;">
      <div style="color:#333; font-size:13px;">
        plan + doc contents<br/>+ fields
      </div>
      <div style="color:#4527a0; font-size:20px; padding:0 12px;">&#10132;</div>
      <div style="background:white; border:1px solid #b39ddb; border-radius:6px;
                  padding:6px 12px; font-size:13px; color:#333;">
        <code>{locator: value}</code> JSON
      </div>
    </div>
  </div>
  <div style="text-align:center; color:#5e35b1; font-size:22px; line-height:1;">&#9660;</div>

  <!-- fill_field -->
  <div style="text-align:center; margin-top:4px;">
    <div style="display:inline-block; background:#fce4ec; border:2px solid #c62828;
                border-radius:24px; padding:8px 24px; font-weight:bold; color:#c62828;">
      fill_field() for each
    </div>
  </div>

  <!-- Legend -->
  <div style="margin-top:16px; padding:10px 14px; background:#f5f5f5; border-radius:8px;
              border:1px solid #ddd; font-size:12px; color:#555;">
    <b>Key idea:</b> The <b>system prompt</b> (dashed boxes) controls <i>what kind of task</i> the LLM performs.
    Same model, same API call structure &mdash; but phase 1 produces a plan while phase 3 produces field values,
    because each gets a different system prompt.
  </div>

</div>
'''))


### 🎯 Task 2 — Build the plan-phase user message

In [ ]:
# Load guide catalog from external file
def load_guide_catalog(guides_folder):
    """Load guide descriptions from guide_catalog.json."""
    catalog_path = Path(guides_folder) / "guide_catalog.json"
    if catalog_path.exists():
        with open(catalog_path) as f:
            raw = json.load(f)
        # Flatten to {name: "description. Use when: ..."} for prompt building
        return {
            name: f"{entry['description']} Use when: {entry['use_when']}"
            for name, entry in raw.items()
        }
    return {}

GUIDE_CATALOG = load_guide_catalog(f"{REPO}/guides")
print(f"Loaded {len(GUIDE_CATALOG)} guide descriptions")


PLAN_SYSTEM_PROMPT = """\
    You are an expert Swiss tax accountant AI.
    You are planning how to fill one page of a Zurich tax return.

    You will see:
    1. The fields on the current form page (locators, labels, types)
    2. A catalog of available documents (names and descriptions only -- NOT their content)
    3. A catalog of available tax guides (names and descriptions -- NOT their content)
    4. What you have already filled on previous pages

    Your job: decide which documents AND which guides you need to read, and outline your strategy for filling the fields.

    ## Response Format (JSON only, no other text)
    {
      "_reasoning": "Your step-by-step reasoning about what this page needs",
      "documents_needed": ["filename1.txt", "filename2.csv"],
      "guides_needed": ["guide-name"],
      "strategy": "Brief description of how you will fill each field",
      "fields_to_fill": ["field-locator-1", "field-locator-2"],
      "ask_user_needed": false
    }

    ## Rules
    - Only request documents that are relevant to the current page's fields.
    - Request guides whenever a field involves a CALCULATION, THRESHOLD, or SPECIAL RULE (e.g. deductions, medical expenses, commuting costs).
    - If no fields need filling (e.g. attachments, review), return an empty documents_needed list.
    - Be specific in your strategy: mention which document contains which value.
    - Do NOT guess document content -- you can only see names and descriptions.
"""


def build_plan_user_message(page_name, fields, catalog, guides_list, filled_history):
    """Build the user message for the PLAN phase."""
    # Filter to fillable fields only
    fillable = [f for f in fields if f.get("type") not in ("button", "text")]
    fields_desc = json.dumps(fillable, indent=2)

    catalog_text = "\n".join(
        f"- {d['filename']}: {d['description']}" for d in catalog
    )

    # Build guide catalog with descriptions so the LLM can make informed selections
    guides_text_lines = []
    for g in guides_list:
        desc = GUIDE_CATALOG.get(g, "")
        if desc:
            guides_text_lines.append(f"- {g}: {desc}")
        else:
            guides_text_lines.append(f"- {g}")
    guides_text = "\n".join(guides_text_lines) if guides_text_lines else "(no guides available)"

    history_lines = []
    for prev_page, prev_fields in filled_history.items():
        entries = ", ".join(f"{k}={v}" for k, v in prev_fields.items() if not k.startswith("_"))
        if entries:
            history_lines.append(f"  {prev_page}: {entries}")
    history_text = "\n".join(history_lines) if history_lines else "  (none yet)"

    # ============================================================
    # 🎯 TODO: Task 2 — assemble the user message for the PLAN phase.
    #
    # Ready-made variables:
    #   page_name     name of the current form page
    #   fields_desc   JSON string of the fillable fields (buttons already removed)
    #   catalog_text  "- filename: description" per document  (names only — no content!)
    #   guides_text   "- guide: description" per guide
    #   history_text  what was filled on previous pages
    #
    # Write ONE f-string with these sections, in this order:
    #   ## Current Page: <page_name>
    #   ### Fields on this page:                        -> fields_desc
    #   ### Available Documents (names and descriptions only):  -> catalog_text
    #   ### Available Tax Guides (names and descriptions):      -> guides_text
    #   ### Previously Filled Fields:                   -> history_text
    # and finish with the instruction:
    #   "Analyze the fields above. Decide which documents AND guides you need to read
    #    and outline your strategy. Respond with JSON only."
    #
    # Why does the wording matter? The system prompt tells the model WHAT KIND of task
    # this is; this user message is the only place it learns about THIS page.
    # ============================================================
    user_msg = f"""..."""   # 🎯 TODO: replace with your message

    return user_msg

In [ ]:
#@title ✅ Checkpoint — does your plan-phase message contain everything? { display-mode: "form" }
# Builds one message from tiny hard-coded inputs and checks that every ingredient made it in. No LLM call.
_fields = [
    {"type": "input", "locator": "field-income-employment-bruttolohn", "label": "1. Gross Salary (CHF)", "value": "", "required": True},
    {"type": "button", "locator": "btn-nav-deductions", "label": "Continue to Deductions"},
]
_catalog = [{"filename": "lohnausweis.txt", "description": "Salary certificate from the employer"}]
_guides = ["policy_kb"]
_history = {"personal": {"field-personal-main-name": "Anna Meier"}}

_msg = build_plan_user_message("income", _fields, _catalog, _guides, _history)
_checks = {
    "names the current page":                      "income" in _msg,
    "lists the fillable field locator":            "field-income-employment-bruttolohn" in _msg,
    "leaves buttons out":                          "btn-nav-deductions" not in _msg,
    "lists the document NAME and description":     "lohnausweis.txt" in _msg and "Salary certificate" in _msg,
    "lists the guide":                             "policy_kb" in _msg,
    "shows what was filled on previous pages":     "Anna Meier" in _msg,
    "asks for JSON only":                          "JSON only" in _msg,
    "no leftover placeholders":                    "🎯" not in _msg and _msg.strip() != "...",
}
for name, ok in _checks.items():
    print(("✅" if ok else "❌"), name)
print("\n🎉 All good — this is exactly what the LLM will read in the Plan phase." if all(_checks.values())
      else "\n👉 Fix the ❌ lines above, then re-run this checkpoint.")


### 🎯 Task 3 — Build the execute-phase user message

In [ ]:
import json, re, time
from pathlib import Path

# What reaches the model in the EXECUTE message is capped per item. The caps are
# guard rails, not a budget: nothing in the five personas is cut. The largest
# document is a bank statement at 3.3k characters of raw CSV plus 9k of the JSON
# rows Task 4a appends (12.3k together); the largest guide is 3.9k once
# load_all_guides() has stripped the HTML. _clip() prints a warning if a cap ever
# does bite, so a truncation never goes unnoticed.
MAX_DOC_CHARS = 16000         # per document: raw content + the "[Structured data]" block
MAX_STRUCTURED_CHARS = 10000  # the "[Structured data]" JSON block alone
MAX_GUIDE_CHARS = 6000        # per guide (plain text)

def _clip(text, limit, label):
    """Cut `text` to `limit` characters, and say so when it happens."""
    if len(text) <= limit:
        return text
    print(f"    ⚠ {label}: {len(text)} chars, only the first {limit} are sent to the model")
    return text[:limit] + "\n[... truncated ...]"

EXECUTE_SYSTEM_PROMPT = """\
    You are an expert Swiss tax accountant AI filling a Zurich tax return.

    You will see:
    1. Your own plan (reasoning and strategy from the planning phase)
    2. The actual content of the documents you requested
    3. The fields on the current form page
    4. Tax guides with rules and thresholds
    5. What you already filled on previous pages

    Your task: fill the fields on this page using the document content and your plan.

    ## Response Format (JSON only, no other text)
    Return a JSON object mapping field locators to values:
    {"_thought": "reasoning here", "field-locator-1": "value1", "field-locator-2": "value2"}

    ## Rules
    - ONLY use locators shown in the current page fields -- NEVER guess locators.
    - If a field already has the correct value, do NOT include it.
    - If a field is required but you have no data, skip it (don't make up values).
    - If no fields apply, return {}.
    - Do NOT fill fields with empty strings, "None", "0", or placeholder text.
    - Include a "_thought" key explaining your reasoning.
"""

# Section-specific prompts (appended to EXECUTE_SYSTEM_PROMPT per page)
SECTION_PROMPTS = {
    "personal": """\
        ## Personal Details
        - Name, Address, DOB, AHV: Extract from profile.json.
        - Parse address into street, streetNumber, zip, city.
        - Marital status: "ledig", "verheiratet", "geschieden", etc.
        - Occupation & Employer: From the Lohnausweis header.
        - Children: Fill BOTH name AND dateOfBirth for each child in profile.json. For the `relationship` field, always use the exact value "child"
          (NOT "Kind", "Tochter", "Sohn", "daughter", or "son" — the form only
          accepts "child"). If profile.json has N children but the form shows fewer rows, include "_rows_needed": N in your response.
    """,

    "income": """\
        ## Income Section
        - Bruttolohn = Lohnausweis field 1 (gross salary).
        - AHV/IV/EO = Lohnausweis field 9.
        - BVG = Lohnausweis field 10.
        - Dienstaltersgeschenke (field 6) are ALREADY INCLUDED in Bruttolohn -- do NOT add separately.
        - Interest income: sum ALL "Zinsabschluss" entries from bank statement.
        - Dividends: declare the GROSS amount from the securities/broker statement.
          Foreign-currency dividends must be converted to CHF per the policy guide;
          if the exchange rate is not in the documents, ask the taxpayer.
        - Self-employment: if ANY freelance/Nebenerwerb documents exist (invoices,1099s, freelance folder), set `selfemployment.enabled` to True AND fill
          the income fields. If no freelance documents exist, leave it False.
        - For married couples with two Lohnausweise: check locator names first.
          If the form has *_partner fields, fill each spouse separately.
          Otherwise, SUM the two Lohnausweise values into the single fields.
        - AHV/IV/EO/ALV contributions: Lohnausweis field 9, MAIN line only
          (the "Beiträge AHV/IV/EO/ALV" line). Do NOT include the NBU sub-line.
        - For married couples: SUM the main-line AHV values of both spouses.
    """,

    "deductions": """\
        ## Deductions Section
        - Verpflegung (meals): Lohnausweis field 14.1 Ja = CHF 1,600; Nein = CHF 3,200.
          Part-time: prorate by Pensum (e.g. 60% + "Nein" = 1920).
        - Berufsauslagen (professional expenses): `flatrate.amount` - compute it from the
          Berufsauslagen-Pauschale rule in the policy guide (policy_kb, Deductions), based on
          Lohnausweis field 12 (Nettolohn). The rule applies PER PERSON: for joint filers,
          compute each spouse's Pauschale separately (each one clamped) and fill the SUM.
          Leave `weitereberufsauslagen` empty unless there are documented further
          professional expenses beyond the Pauschale.
        - Fahrkosten (commuting): start from the public-transport pass receipt (ZVV), then
          check the guide catalog for commuting flat rates (e.g. the Velo-Pauschale guide)
          and add whatever the guide says applies to this taxpayer.
        - Pillar 3a: max CHF 7,056 (employed with BVG).
        - Donations: min CHF 100, look for "Spende" in bank statement.
        - Schuldzinsen: sum ALL mortgage interest from bank statement.
        - Weiterbildung: only if course is related to current profession.
        - Zweiverdienerabzug: if both spouses employed, fill exactly CHF 5,900 per the policy guide (policy_kb, Deductions). Do NOT apply a percentage formula.
        - Kinderbetreuungskosten: cap per child per year is CHF 10,100 per the policy guide (policy_kb, Deductions). If receipts exceed the cap, use the cap.
        - Berufsauslagen Pauschale vs. effektiv: these are ALTERNATIVES, not cumulative.
          Choose one mode per person:
          - Pauschale mode (default): the flat rate per the policy guide.
          - Effektiv mode: sum of documented receipts (Physiosuisse, Fachliteratur,
            EDV-Hardware, etc.), only if the total EXCEEDS that person's Pauschale.
          Do NOT add individual receipts on top of the Pauschale.
          For married couples: decide mode per spouse independently, then sum.
        - AHV/IV/EO/ALV contributions: field 9 MAIN line only, NOT NBU sub-line.
        - Medical expenses (Krankheitskosten): DEDUCTIBLE ONLY above 5% of net income.
          STEP 1: Compute net income = total gross income − social insurance − BVG − professional expenses.
          STEP 2: Compute threshold = net_income × 0.05.
          STEP 3: If medical_total ≤ threshold, fill 0 (zero). Do NOT fill the raw receipt total.
          STEP 4: Only if medical_total > threshold, fill (medical_total − threshold).
    """,

    "wealth": """\
        ## Wealth Section
        - Bank balance: LAST line of bank statement CSV (Dec 31 balance).
        - Bank name: read from the CSV metadata header (lines starting with '#' at top
          of file). Use the bank name as written there.
        - Bank interest: sum ALL "Zinsabschluss" entries.
        - Securities: market value as of Dec 31 (convert foreign currency to CHF as for dividends).
          If dividends are declared, the securities table MUST contain at least one row:
          one aggregate row per depot is fine (name = depot/broker, value = total market
          value in CHF, grossReturn = total gross dividends). Leave isin/quantity empty for
          an aggregate row.
        - Real estate: use Steuerwert from Gemeinde assessment, NOT market value.
        - In Bank Accounts Include "_rows_needed": N if rows need adding.
        - In Debts field creditor = "Hypothek" unless the property assessment or a loan document explicitly names the lender.
          The debt amount is the OUTSTANDING MORTGAGE BALANCE (e.g. "Hypothekarschuld" on the
          property assessment), NOT the yearly interest - the interest goes to deductions.schuldzinsen.
        - Alimony received: declare the amount legally ordered per the Scheidungsurteil
             (decree), NOT the amount actually received in cash. If the decree specifies
             monthly amounts, multiply by the number of months the decree was in force
             during the tax year.

        ### Dynamic Rows
        - Include "_rows_needed": N if rows need adding.
        - If rows already exist, fill directly.
    """,

    "attachments": "## Attachments\nFile uploads cannot be filled programmatically. Return {}.",
    "review": "## Review\nThis is the final review page. Return {}.",
}


def build_execute_user_message(page_name, fields, plan, doc_contents, guide_contents, filled_history):
    """Build the user message for the EXECUTE phase."""
    fillable = [f for f in fields if f.get("type") not in ("button", "text")]
    fields_desc = json.dumps(fillable, indent=2)

    docs_text = "\n\n".join(
        f"=== {name} ===\n{_clip(content, MAX_DOC_CHARS, f'document {name}')}"
        for name, content in doc_contents.items()
    )
    guides_text = "\n\n".join(
        f"=== {name} ===\n{_clip(content, MAX_GUIDE_CHARS, f'guide {name}')}"
        for name, content in guide_contents.items()
    )

    history_lines = []
    for prev_page, prev_fields in filled_history.items():
        entries = ", ".join(f"{k}={v}" for k, v in prev_fields.items() if not k.startswith("_"))
        if entries:
            history_lines.append(f"  {prev_page}: {entries}")
    history_text = "\n".join(history_lines) if history_lines else "  (none yet)"

    plan_text = json.dumps(plan, indent=2) if isinstance(plan, dict) else str(plan)

    # ============================================================
    # 🎯 TODO: Task 3 — assemble the user message for the EXECUTE phase.
    #
    # Ready-made variables:
    #   plan_text     the plan from phase 1 (JSON string)
    #   docs_text     the ACTUAL content of the documents the plan requested
    #   guides_text   the guide texts the plan requested ("" if none)
    #   history_text  what was filled on previous pages
    #   page_name, fields_desc   as in the plan message
    #
    # Write ONE f-string with these sections, in this order:
    #   ## Your Plan (from planning phase)               -> plan_text
    #   ## Document Contents (the actual data you requested)  -> docs_text
    #   ## Tax Guides                                    -> guides_text (only if not empty)
    #   ## Previously Filled Fields                      -> history_text
    #   ## Current Page: <page_name>  +  ### Fields on this page:  -> fields_desc
    # and finish with:
    #   "Now fill the fields. Return a JSON object mapping locators to values.
    #    If no fields apply, return empty braces."
    # ============================================================
    user_msg = f"""..."""   # 🎯 TODO: replace with your message

    return user_msg

In [ ]:
#@title ✅ Checkpoint — does your execute-phase message contain everything? { display-mode: "form" }
# Builds one message from tiny hard-coded inputs and checks that every ingredient made it in. No LLM call.
_fields = [{"type": "input", "locator": "field-income-employment-bruttolohn", "label": "1. Gross Salary (CHF)", "value": "", "required": True}]
_plan = {"_reasoning": "PLAN-REASONING-MARKER", "documents_needed": ["lohnausweis.txt"], "strategy": "read field 1"}
_docs = {"lohnausweis.txt": "1. Lohn (brutto)    CHF 142,000.00"}
_guides = {"policy_kb": "Berufsauslagen Pauschale: 3% of Nettolohn"}
_history = {"personal": {"field-personal-main-name": "Anna Meier"}}

_msg = build_execute_user_message("income", _fields, _plan, _docs, _guides, _history)
_checks = {
    "includes the plan":                      "PLAN-REASONING-MARKER" in _msg,
    "includes the document CONTENT":          "142,000.00" in _msg,
    "includes the guide text":                "Berufsauslagen Pauschale" in _msg,
    "shows what was filled on previous pages":"Anna Meier" in _msg,
    "names the current page":                 "income" in _msg,
    "lists the field locator":                "field-income-employment-bruttolohn" in _msg,
    "asks for a JSON answer":                 "JSON" in _msg,
    "no leftover placeholders":               "🎯" not in _msg and _msg.strip() != "...",
}
for name, ok in _checks.items():
    print(("✅" if ok else "❌"), name)
print("\n🎉 All good — this is exactly what the LLM will read in the Execute phase." if all(_checks.values())
      else "\n👉 Fix the ❌ lines above, then re-run this checkpoint.")


### 🧰 Functions you'll use in `think_step` and `think_loop`

Here are the key functions you haven't seen yet. You'll call these in the cells below:

```python
# ── LLM calls ──
llm.chat_json(system, user, max_tokens=4096) -> dict
    # Like llm.chat(), but parses the response as a JSON dict.
    # Use this when you need structured output (plans, field mappings).

# ── Server tools ──
server.scan_page() -> dict
    # Returns {"page_name": ..., "elements": [...]} for the page the form is on.

server.read_document(filename) -> dict
    # Returns {"content": "...", "filename": "..."}
    # Reads a document from the persona's folder.

server.fill_field(locator, value) -> dict
    # Returns {"success": True/False, ...}
    # Enters a value into the form field identified by locator.

# ── Helper (defined in next cell) ──
fill_page(server, mapping, page_name, filled_history) -> dict
    # Calls server.fill_field() for each {locator: value} in mapping.
    # Updates filled_history and returns the dict of successfully filled fields.
```


### 🎯 Task 4 — Implement `think_step`

**Before you start coding `think_step`**, look back at the **architecture diagram** above. The function you're about to write implements all three boxes:

1. **Plan phase** — call the LLM with `PLAN_SYSTEM_PROMPT` + `build_plan_user_message()`
2. **Selective retrieval** — read only the documents the plan requested
3. **Execute phase** — call the LLM with `EXECUTE_SYSTEM_PROMPT` + `build_execute_user_message()`

Each phase is a separate `llm.chat_json()` call with a different system prompt.

In [ ]:
def think_step(server, llm, page_name, elements, catalog, all_guides, filled_history, section_prompts):
    """Plan-then-Execute reasoning for one page.

    Phase 1 (Plan):    LLM sees field labels + doc catalog -> outputs a plan
    Retrieval:         read ONLY the documents/guides the plan asked for
    Phase 2 (Execute): LLM sees plan + selected doc contents -> outputs field mappings

    Returns
    -------
    tuple of (plan_dict, mapping_dict)
        plan_dict:    the plan JSON from phase 1
        mapping_dict: {locator: value} from phase 2
    """
    fields = [e for e in elements if e.get("type") not in ("button", "text")]
    if not fields:
        return {}, {}

    guides_list = list(all_guides.keys())

    # ════════════════════════════════════════════════════════════
    # PHASE 1: PLAN  (given — one LLM call with the PLAN system prompt)
    # ════════════════════════════════════════════════════════════
    plan_user_msg = build_plan_user_message(
        page_name, elements, catalog, guides_list, filled_history
    )
    plan = llm.chat_json(PLAN_SYSTEM_PROMPT, plan_user_msg)

    print(f"  Plan: need docs {plan.get('documents_needed', [])}")
    print(f"  Strategy: {plan.get('strategy', 'none')}")

    # ════════════════════════════════════════════════════════════
    # SELECTIVE RETRIEVAL
    # ════════════════════════════════════════════════════════════
    # 🎯 TODO: Task 4a — read only the documents the plan requested.
    doc_contents = {}
    for doc_name in ...:   # 🎯 TODO ... <-- FILL THIS IN
        try:
            doc = server.read_document(doc_name)
            content = doc.get("content", "")
            if doc.get("structured"):
                content += "\n\n[Structured data]:\n" + _clip(
                    json.dumps(doc["structured"], indent=2, ensure_ascii=False),
                    MAX_STRUCTURED_CHARS, f"structured data of {doc_name}")
            doc_contents[doc_name] = content
            print(f"    Read: {doc_name} ({len(content)} chars)")
        except Exception as e:
            print(f"    Could not read {doc_name}: {e}")

    # 🎯 TODO: fetch only the guides the plan requested (they are already loaded in `all_guides`).
    guide_contents = {}
    for guide_name in ...:   # 🎯 TODO ... <-- FILL THIS IN
        if guide_name in all_guides:
            guide_contents[guide_name] = all_guides[guide_name]
            print(f"    Loaded guide: {guide_name}")
        else:
            # Fuzzy match: try partial name matching
            for key in all_guides:
                if guide_name.lower() in key.lower() or key.lower() in guide_name.lower():
                    guide_contents[key] = all_guides[key]
                    print(f"    Loaded guide (fuzzy): {key}")
                    break

    # ════════════════════════════════════════════════════════════
    # PHASE 2: EXECUTE
    # ════════════════════════════════════════════════════════════
    section = page_name.split("/")[0]
    # 🎯 TODO: Task 4b — call the LLM to get the field mappings.
    # Use: llm.chat_json(), system_msg, build_execute_user_message()
    # Hint: what should the system prompt be?
    section_prompt = ...   # 🎯 TODO ... <-- FILL THIS IN
    system_msg = ...   # 🎯 TODO ... <-- FILL THIS IN
    mapping = ...   # 🎯 TODO ... <-- FILL THIS IN

    return plan, mapping


### 🎯 Task 5 — Test `think_step` on one page

In [ ]:
# Test think_step on the Income page (MockBridge)

from mcp_server.bridges.mock import MockBridge

test_bridge = MockBridge()
test_bridge.click_element("btn-nav-income")   # navigate with the same tool the agent uses

test_server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=test_bridge,
)

test_catalog = build_document_catalog(test_server)

def load_all_guides(guides_folder):
    """Read every guide once, as plain text, keyed by topic (the file stem).

    Ten of the eleven guides are HTML articles. Sent raw, their <style> block would
    eat most of the characters that reach the model — so strip them here with the
    same html_to_text() you used in Section 3.
    """
    guides = {}
    folder = Path(guides_folder)
    if not folder.exists():
        return guides
    for f in sorted(folder.iterdir()):
        if f.suffix in (".html", ".txt", ".md"):
            guides[f.stem] = html_to_text(f.read_text(encoding="utf-8"))
    return guides

all_guides = load_all_guides(f"{REPO}/guides")

# 🎯 TODO: Task 5 — get the current page data from the server.
# Which tool from Section 4 returns all fields on the current page? (hint: test_server.<tool>())
page_data = ...
fillable = [e for e in page_data["elements"] if e.get("type") not in ("button", "text")]

print(f"Testing think_step on: {page_data['page_name']} ({len(fillable)} fillable fields)")

plan, mapping = think_step(
    server=test_server,
    llm=llm,
    page_name=page_data["page_name"],
    elements=page_data["elements"],
    catalog=test_catalog,
    all_guides=all_guides,
    filled_history={},
    section_prompts=SECTION_PROMPTS,
)

print(f"Plan returned {len(plan.get('documents_needed', []))} documents")
print(f"Mapping returned {len([k for k in mapping if not k.startswith('_')])} fields")

In [ ]:
#@title Display think_step results & ground truth comparison { display-mode: "form" }
from IPython.display import display, HTML

# Colors follow Colab's theme variables so the output is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
RESULT_STYLE = """
<style>
  .ts-card {
    border: 2px solid #2d7d2d; border-radius: 8px; padding: 14px; margin: 10px 0;
    font-family: sans-serif; color: var(--colab-primary-text-color, inherit);
    background: var(--colab-highlighted-surface-color, rgba(45,125,45,.08));
  }
  .ts-card.blue { border-color: #4a90d9; background: var(--colab-highlighted-surface-color, rgba(74,144,217,.08)); }
  .ts-card h4 { margin: 0 0 8px 0; color: #2d7d2d; }
  .ts-card.blue h4 { color: #4a90d9; }
  .ts-card table { border-collapse: collapse; width: 100%; }
  .ts-card th { padding: 6px 8px; text-align: left; color: #fff; background: #2d7d2d; }
  .ts-card.blue th { background: #4a90d9; }
  .ts-card td { padding: 4px 8px; border-bottom: 1px solid var(--colab-border-color, rgba(128,128,128,.35)); }
  .ts-card code { color: inherit; background: rgba(128,128,128,.15); padding: 1px 4px; border-radius: 3px; }
  .ts-card .ok { color: #2d7d2d; }
  .ts-card .bad { color: #e53935; }
  .ts-card .warn { color: #ef8c00; }
  .ts-card .muted { color: var(--colab-secondary-text-color, #888); }
</style>
"""

# Display field mappings
thought = mapping.get("_thought", "")
rows = "".join(
    f'<tr><td><code>{k}</code></td><td>{v}</td></tr>'
    for k, v in mapping.items() if not k.startswith("_")
)
display(HTML(RESULT_STYLE + f'''
<div class="ts-card">
    <h4>Field Mappings from LLM</h4>
    {"<div style='margin-bottom:8px;'><b>Reasoning:</b> " + thought + "</div>" if thought else ""}
    <table>
    <tr><th>Locator</th><th>Value</th></tr>
    {rows if rows else "<tr><td colspan='2' class='muted'><i>No mappings returned</i></td></tr>"}
    </table>
</div>
'''))

# Ground truth comparison
gt_path = Path(f"{REPO}/personas/{PERSONA}/ground_truth.json")
if gt_path.exists():
    with open(gt_path) as f:
        ground_truth = json.load(f)

    def flatten_gt(section_data, prefix="field-income"):
        flat = {}
        for key, val in section_data.items():
            if isinstance(val, dict):
                for subkey, subval in val.items():
                    flat[f"{prefix}-{key}-{subkey}"] = str(subval)
            else:
                flat[f"{prefix}-{key}"] = str(val)
        return flat

    gt_income = flatten_gt(ground_truth.get("income", {}))
    agent_values = {k: str(v) for k, v in mapping.items() if not k.startswith("_")}

    all_locators = sorted(set(list(gt_income.keys()) + list(agent_values.keys())))
    comp_rows = ""
    for loc in all_locators:
        expected = gt_income.get(loc, "")
        actual = agent_values.get(loc, "")
        if expected and actual:
            match = abs(float(expected) - float(actual)) < 1 if expected.replace(".", "").isdigit() and actual.replace(".", "").isdigit() else expected.lower() == actual.lower()
            css = "ok" if match else "bad"
            icon = "&#10003;" if match else "&#10007;"
        elif expected and not actual:
            css = "warn"
            icon = "&#8709;"
        else:
            css = "muted"
            icon = "?"
        comp_rows += (
            f'<tr class="{css}">'
            f'<td><code>{loc}</code></td>'
            f'<td>{expected or "<i>-</i>"}</td>'
            f'<td>{actual or "<i>not filled</i>"}</td>'
            f'<td style="text-align:center;">{icon}</td></tr>'
        )

    display(HTML(RESULT_STYLE + f'''
    <div class="ts-card blue">
        <h4>Ground Truth Comparison (Income)</h4>
        <table>
        <tr><th>Locator</th><th>Expected</th><th>Agent</th><th style="text-align:center;">Match</th></tr>
        {comp_rows}
        </table>
    </div>
    '''))
else:
    print("Ground truth file not found — skipping comparison.")

**✅ Checkpoint:** for Anna Meier the table above should show these four income values (small rounding differences are fine):

| Locator | Expected |
|---|---|
| `field-income-employment-bruttolohn` | 142000 |
| `field-income-employment-ahvcontributions` | 7526.60 |
| `field-income-employment-bvgcontributions` | 8400 |
| `field-income-investment-interest` | 12.50 |

If they match, you're ready to visualize the plan and build the full loop.

If the values look wrong, check:
- Is `build_plan_user_message` sending the document catalog correctly?
- Is `build_execute_user_message` including the actual document content?
- Is the plan requesting the right documents?

## 7. 🎨 Plan Visualization

One advantage of the Plan-Execute split: we can **visualize** the agent's decision-making before it acts. This makes the agent's reasoning transparent and debuggable.

In [ ]:
#@title Visualize the plan from the test { display-mode: "form" }
# Pre-built visualization function
from IPython.display import display, HTML

# Colors follow Colab's theme variables so the card is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
PLAN_CARD_STYLE = """
<style>
  .plan-viz {
    border: 2px solid #4a90d9; border-radius: 10px;
    padding: 16px; margin: 12px 0; font-family: sans-serif;
    color: var(--colab-primary-text-color, inherit);
    background: var(--colab-highlighted-surface-color, rgba(74,144,217,.08));
  }
  .plan-viz h3 { color: #4a90d9; margin-top: 0; }
  .plan-viz .panel {
    border: 1px solid var(--colab-border-color, rgba(128,128,128,.35));
    border-radius: 6px; padding: 12px; margin: 8px 0;
  }
  .plan-viz blockquote {
    border-left: 3px solid #4a90d9; padding-left: 12px; margin: 8px 0;
  }
  .plan-viz code {
    color: inherit;
    background: rgba(128,128,128,.15);
    padding: 2px 6px; margin: 2px; border-radius: 3px;
  }
  .plan-viz .doc { margin: 2px 0; }
  .plan-viz .doc.selected { color: #2d7d2d; font-weight: bold; }
  .plan-viz .doc.skipped { color: var(--colab-secondary-text-color, #888); }
  .plan-viz .muted {
    color: var(--colab-secondary-text-color, #888); font-style: italic;
  }
</style>
"""

def visualize_plan(page_name, plan, doc_catalog):
    """Render the agent's plan as a styled HTML card."""
    selected = set(plan.get("documents_needed", []))

    docs_html = ""
    for doc in doc_catalog:
        is_selected = doc["filename"] in selected
        icon = "&#9745;" if is_selected else "&#9744;"
        css = "selected" if is_selected else "skipped"
        docs_html += (
            f'<div class="doc {css}">'
            f'{icon} <code>{doc["filename"]}</code> — {doc["description"]}</div>'
        )

    fields_html = " ".join(f"<code>{f}</code>" for f in plan.get("fields_to_fill", []))

    reasoning = plan.get("_reasoning", "No reasoning provided")
    strategy = plan.get("strategy", "No strategy provided")

    html = f"""
    <div class="plan-viz">
        <h3>Plan: {page_name}</h3>
        <div class="panel">
            <b>Reasoning:</b>
            <blockquote>{reasoning}</blockquote>
        </div>
        <div class="panel">
            <b>Documents selected:</b><br/>
            {docs_html}
        </div>
        <div class="panel">
            <b>Strategy:</b> {strategy}
        </div>
        <div class="panel">
            <b>Fields to fill:</b><br/>{fields_html if fields_html else "<span class='muted'>not specified</span>"}
        </div>
    </div>
    """
    display(HTML(PLAN_CARD_STYLE + html))

# Render the plan produced by the think_step test above
if plan:
    visualize_plan(page_data["page_name"], plan, test_catalog)
else:
    print("No plan to visualize yet. Run the think_step test cell first.")

## 8. 🔁 Think Loop — Full Pipeline

Now we wire `think_step` into the full navigation loop. The code drives page-by-page navigation; your `think_step` handles the reasoning on each page.

The loop: **scan** → **plan** → **retrieve** → **execute** → **fill** → **next page** → repeat until review → **submit**.

In [ ]:
#@title Helpers for think_loop: dynamic rows, filling, visualization { display-mode: "form" }
from IPython.display import display, HTML

def add_rows_if_needed(server, elements, num_rows_needed=0):
    """Click 'Add Row' buttons to create dynamic table rows."""
    add_btns = [e for e in elements
                if e.get("type") == "button"
                and "add-row" in (e.get("locator") or "").lower()]
    if not add_btns:
        return None

    existing_rows = set()
    for e in elements:
        loc = e.get("locator") or ""
        match = re.search(r'-(\d+)-', loc)
        if match:
            existing_rows.add(int(match.group(1)))

    rows_to_add = max(0, (num_rows_needed or 1) - len(existing_rows))
    if rows_to_add > 0:
        for _ in range(rows_to_add):
            for btn in add_btns:
                loc = btn.get("locator")
                if loc:
                    print(f"  Adding row: {loc}")
                    server.click_element(loc)
        return server.scan_page()
    return None


# ── Helper: fill fields from mapping ─────────────────────────

def fill_page(server, mapping, page_name, filled_history):
    """Execute fill_field for each locator:value in the mapping."""
    filled = {}
    thought = mapping.pop("_thought", None)
    if thought:
        print(f"  Reasoning: {thought}")

    for locator, value in mapping.items():
        if locator.startswith("_"):
            continue
        result = server.fill_field(locator, str(value))
        if result.get("success"):
            filled[locator] = value
            print(f"  Filled: {locator} = {value}")
        else:
            print(f"  FAILED: {locator}: {result.get('error', 'unknown')}")

    if filled:
        filled_history[page_name] = filled
    return filled

# ── Visualization helpers for think_loop ──────────────────────
# Colors follow Colab's theme variables so the output is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
LOOP_STYLE = """
<style>
  .loop { font-family: sans-serif; color: var(--colab-primary-text-color, inherit); margin: 4px 0; }
  .loop.header {
    border-left: 4px solid #4a90d9; padding: 8px 14px; margin: 16px 0 8px 0;
    background: var(--colab-highlighted-surface-color, rgba(74,144,217,.08));
  }
  .loop.header b { color: #4a90d9; }
  .loop.submit {
    border: 2px solid #2d7d2d; border-radius: 8px; padding: 14px; margin: 10px 0;
    background: var(--colab-highlighted-surface-color, rgba(45,125,45,.08));
  }
  .loop.submit h4 { color: #2d7d2d; margin: 0; }
  .loop.muted { color: var(--colab-secondary-text-color, #888); font-style: italic; }
  .loop.ok { color: #2d7d2d; }
  .loop.nav { color: #4a90d9; }
  .loop.bad { color: #e53935; }
  .loop.warn { color: #ef8c00; margin-top: 12px; }
</style>
"""

def _loop_html(cls, inner):
    display(HTML(LOOP_STYLE + f'<div class="loop {cls}">{inner}</div>'))

def display_page_header(step, page_name):
    """Show a styled header for the current page."""
    _loop_html("header", f"<b>Page {step}: {page_name}</b>")

def display_submit_result(result):
    """Show submission success/failure."""
    _loop_html("submit", f'<h4>Submitted! Success: {result.get("success")}</h4>')

def display_skip_message():
    """Show message when no fillable fields found."""
    _loop_html("muted", "No fillable fields — skipping")

def display_field_count(count):
    """Show the number of fillable fields."""
    _loop_html("", f"<b>{count}</b> fillable fields")

def display_filled_count(count):
    """Show how many fields were filled."""
    _loop_html("ok", f"<b>{count}</b> fields filled")

def display_empty_mapping():
    """Show message when LLM returns empty mapping."""
    _loop_html("muted", "LLM returned empty mapping — nothing to fill")

def display_navigation(target):
    """Show navigation message."""
    _loop_html("nav", f"&#8594; Navigated to: <b>{target}</b>")

def display_nav_error():
    """Show navigation failure."""
    _loop_html("bad", "<b>Cannot navigate further — stopping</b>")

def display_max_steps(max_steps):
    """Show max steps warning."""
    _loop_html("warn", f"<b>Max steps ({max_steps}) reached without submitting.</b>")

print("Helpers defined")

### 🎯 Task 6 — Wire `think_step` into `think_loop` and run the agent

In [ ]:
def think_loop(server, llm, max_steps=100):
    """Full agent: Plan-Execute on every page with visualization."""

    catalog = build_document_catalog(server)
    all_guides = load_all_guides(f"{REPO}/guides")
    filled_history = {}
    plans = []
    total_fields = 0

    # Auto-navigate away from root/overview
    initial = server.scan_page()
    page_name = initial.get("page_name", "")
    if page_name in ("root", "overview", ""):
        for loc in ["tab-nav-form", "btn-nav-personal", "btn-login"]:
            r = server.click_element(loc)
            if r.get("success") and r.get("new_page"):
                break

    visited = set()
    step = 0

    while step < max_steps:
        step += 1
        page_data = server.scan_page()
        page_name = page_data.get("page_name", "unknown")
        elements = page_data.get("elements", [])

        # Skip non-form pages
        if page_name in ("root", "overview", ""):
            server.click_element("tab-nav-form")
            continue

        # Skip already-visited pages
        if page_name in visited:
            nav = server.click_element("btn-next")
            if not nav.get("new_page"):
                break
            continue
        visited.add(page_name)

        display_page_header(step, page_name)

        # Submit on review page
        if page_name == "review":
            result = server.submit_form()
            display_submit_result(result)
            return {
                "submission": result,
                "filled_history": filled_history,
                "plans": plans,
            }

        # Handle dynamic rows
        fillable = [e for e in elements if e.get("type") not in ("button", "text")]
        has_add_row = any(
            e.get("type") == "button" and "add-row" in (e.get("locator") or "").lower()
            for e in elements
        )
        if not fillable and has_add_row:
            new_scan = add_rows_if_needed(server, elements)
            if new_scan:
                elements = new_scan.get("elements", [])

        fillable = [e for e in elements if e.get("type") not in ("button", "text")]
        if not fillable:
            display_skip_message()
        else:
            display_field_count(len(fillable))

            # ════════════════════════════════════════════════════
            # 🎯 TODO: Task 6 — plug your think_step into the loop.
            # Steps 2 and 3 are given; you write steps 1 and 4.
            # ════════════════════════════════════════════════════

            # Step 1: get (plan, mapping) for this page.
            # think_step needs: server, llm, page_name, elements, catalog, all_guides,
            #                   filled_history, SECTION_PROMPTS
            plan, mapping = {}, {}   # 🎯 TODO: replace with your think_step(...) call

            # Step 2: Visualize
            if plan:
                plans.append({"page": page_name, "plan": plan})
                visualize_plan(page_name, plan, catalog)

            # Step 3: Handle _rows_needed
            rows_needed = int(mapping.pop("_rows_needed", 0))
            if rows_needed > 0:
                new_scan = add_rows_if_needed(server, elements, rows_needed)
                if new_scan:
                    elements = new_scan.get("elements", [])
                    _, mapping = think_step(
                        server, llm, page_name, elements,
                        catalog, all_guides, filled_history, SECTION_PROMPTS,
                    )
                    mapping.pop("_rows_needed", None)

            # Step 4: fill the page and count what was filled.
            if mapping:
                filled = {}   # 🎯 TODO: replace with fill_page(server, mapping, page_name, filled_history)
                total_fields += len(filled)
                display_filled_count(len(filled))
            else:
                display_empty_mapping()

        # Navigate to next page
        nav_result = server.click_element("btn-next")
        if nav_result.get("success") and nav_result.get("new_page"):
            display_navigation(nav_result["new_page"])
        else:
            for nav_target in ["tab-nav-form", "nav-personal", "nav-income",
                               "nav-deductions", "nav-wealth",
                               "nav-attachments", "nav-review"]:
                r = server.click_element(nav_target)
                if r.get("success") and r.get("new_page"):
                    display_navigation(r["new_page"])
                    break
            else:
                display_nav_error()
                break

    display_max_steps(max_steps)
    return {"submission": None, "filled_history": filled_history, "plans": plans}

### 🚀 Run the agent: display UI, init server, run

In [ ]:
# Note: this must stay a single cell. Colab's eval_js() only works within
# the same output cell where the HTML was rendered.

# 🎯 TODO: Choose a persona. Start with anna_meier, then climb the difficulty ladder.
PERSONA = "anna_meier"
# Available: 'anna_meier' (Easy), 'marco_laura_bernasconi' (Medium),
#            'priya_chakraborty' (Hard), 'thomas_elisabeth_widmer' (Very Hard),
#            'yuki_tanaka' (Expert)

# 1. Render the UI
from IPython.display import display, HTML
import os, json, datetime
from google.colab.output import eval_js
import time

BUILD_DIR = f"{REPO}/EnvDemoV1/dist"
with open(os.path.join(BUILD_DIR, "index.html")) as f:
    html_content = f.read()

html_content = html_content.replace(r'"\/assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"./assets/', f'"http://localhost:{PORT}/assets/')
html_content = html_content.replace('"/assets/', f'"http://localhost:{PORT}/assets/')

display(HTML(html_content))
print(f"UI rendered from http://localhost:{PORT}")

time.sleep(1)
eval_js(f'window.TaxPortal.setPersona("{PERSONA}")')

# 2. Init MCP Server with ColabBridge
from mcp_server.server import MCPServer
from mcp_server.bridges.colab import ColabBridge

server = MCPServer(
    persona_folder=f"{REPO}/personas/{PERSONA}",
    guides_folder=f"{REPO}/guides",
    bridge=ColabBridge(),
)
print(f"Server ready for: {PERSONA}")

# 3. Run the agent
print(f"\nStarting agent for {PERSONA}...\n")
agent_result = think_loop(server, llm, max_steps=100)
print("\nDone!")

# 4. Save outputs
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = f"/content/agent_output_{PERSONA}_{timestamp}"
os.makedirs(out_dir, exist_ok=True)

with open(f"{out_dir}/filled_history.json", "w") as f:
    json.dump(agent_result.get("filled_history", {}), f, indent=2, ensure_ascii=False)
with open(f"{out_dir}/plans.json", "w") as f:
    json.dump(agent_result.get("plans", []), f, indent=2, ensure_ascii=False, default=str)
with open(f"{out_dir}/submission.json", "w") as f:
    json.dump(agent_result.get("submission", {}), f, indent=2, ensure_ascii=False, default=str)

print(f"Output saved to: {out_dir}")

## 9. 📊 Score your agent

The scorer compares the submitted return with the persona's answer key **field by field**. Each field has a
declared type (number, date, enum, free text, …) that decides how it is compared — numbers get a 1 CHF
tolerance, `ledig` counts as `single`, free text goes through a fuzzy match and, if that is inconclusive,
an LLM judge. Fields you filled that the answer key leaves empty count as **extra** and cost half a point.
The card tells you, for every field, *which rule* decided the verdict — and a collapsed table shows
everything you got right.


In [ ]:
#@title Scoring — typed scorer (scripts/eval_lib.py) { display-mode: "form" }
!pip install -q rapidfuzz
# All comparison logic lives in scripts/eval_lib.py (see tests/test_eval_lib.py for the spec).
# This cell only loads the ground truth, calls score() and renders the result.
import sys, json
from pathlib import Path
from IPython.display import display, HTML

if REPO not in sys.path:
    sys.path.insert(0, REPO)
from scripts.eval_lib import score, load_schema, extract_submission_data

# Colors follow Colab's theme variables so the output is readable in both
# light and dark mode. Fallbacks are for non-Colab renderers.
SCORE_STYLE = """
<style>
  .score { font-family: sans-serif; color: var(--colab-primary-text-color, inherit);
           border: 2px solid; border-radius: 10px; padding: 16px; margin: 12px 0; }
  .score h3 { margin: 0 0 10px 0; }
  .score h3 small { font-size: 0.7em; color: var(--colab-secondary-text-color, #888); }
  .score .totals { display: flex; gap: 20px; margin-bottom: 8px; flex-wrap: wrap; }
  .score .bar { background: var(--colab-border-color, rgba(128,128,128,.25)); border-radius: 6px;
                height: 20px; overflow: hidden; margin: 8px 0; }
  .score .bar div { height: 100%; border-radius: 6px; }
  .score table { border-collapse: collapse; width: 100%; margin-top: 10px; }
  .score th { padding: 6px 8px; text-align: left; color: #fff; background: #666; }
  .score th.green { background: #2d7d2d; }
  .score td { padding: 4px 8px; border-bottom: 1px solid var(--colab-border-color, rgba(128,128,128,.35)); }
  .score code { color: inherit; background: rgba(128,128,128,.15); padding: 1px 4px; border-radius: 3px; }
  .score .tier { color: var(--colab-secondary-text-color, #888); font-size: 0.85em; }
  .score tr.wrong td:first-child { border-left: 4px solid #e53935; }
  .score tr.missing td:first-child { border-left: 4px solid #ef8c00; }
  .score tr.extra td:first-child { border-left: 4px solid #7e57c2; }
  .score tr.correct td:first-child { border-left: 4px solid #2d7d2d; }
  .score .ok { color: #2d7d2d; } .score .bad { color: #e53935; }
  .score .warn { color: #ef8c00; } .score .extra { color: #7e57c2; }
  .score details { margin-top: 12px; }
  .score summary { cursor: pointer; color: #2d7d2d; font-weight: bold; }
</style>
"""

def score_submission(repo, persona, agent_result, judge=None):
    gt_path = Path(f"{repo}/personas/{persona}/ground_truth.json")
    if not gt_path.exists():
        display(HTML(f'<div class="score bad" style="border-color:#e53935;"><b>No ground_truth.json for {persona}</b></div>'))
        return None

    with open(gt_path) as f:
        ground_truth = json.load(f)
    submission_data = extract_submission_data(agent_result)
    report = score(ground_truth, submission_data, load_schema(f"{repo}/personas/field_schema.json"),
                   judge=judge, persona=persona)

    rows = []
    for r in report.problems():
        got = "—" if r.sub in (None, "") else r.sub
        exp = "—" if r.gt in (None, "") else r.gt
        rows.append(
            f'<tr class="{r.verdict}">'
            f'<td><code>{r.path}</code></td><td>{r.verdict}</td><td>{got}</td><td>{exp}</td>'
            f'<td class="tier">{r.type} · {r.tier}</td></tr>'
        )

    score_color = "#2d7d2d" if report.score >= 80 else "#ef8c00" if report.score >= 50 else "#e53935"
    table = ""
    if rows:
        table = ('<table><tr><th>Field</th><th>Verdict</th><th>Got</th><th>Expected</th><th>Decided by</th></tr>'
                 + "".join(rows) + '</table>')
    untyped = (f'<div class="warn" style="margin-top:6px;">⚠ {len(report.untyped)} field(s) have no type in field_schema.json</div>'
               if report.untyped else "")

    # Collapsed by default: everything the agent got right, next to the answer key.
    correct_rows = "".join(
        f'<tr class="correct"><td><code>{r.path}</code></td>'
        f'<td>{"—" if r.sub in (None, "") else r.sub}</td><td>{r.gt}</td>'
        f'<td class="tier">{r.type} · {r.tier}</td></tr>'
        for r in report.rows if r.verdict == "correct"
    )
    details = ""
    if correct_rows:
        details = (
            f'<details><summary>▸ Show the {report.correct} correct fields (agent value vs. answer key)</summary>'
            f'<table><tr><th class="green">Field</th><th class="green">Got</th>'
            f'<th class="green">Expected</th><th class="green">Decided by</th></tr>{correct_rows}</table></details>'
        )

    display(HTML(SCORE_STYLE +
        f'<div class="score" style="border-color:{score_color};">'
        f'<h3 style="color:{score_color};">Score: {report.score:.1f}% '
        f'<small>({report.correct}/{report.total} correct, '
        f'{report.extra} extra × −0.5; raw {report.raw_score:.1f}%)</small></h3>'
        f'<div class="totals">'
        f'<span class="ok"><b>{report.correct}</b> Correct</span>'
        f'<span class="bad"><b>{report.wrong}</b> Wrong</span>'
        f'<span class="warn"><b>{report.missing}</b> Missing</span>'
        f'<span class="extra"><b>{report.extra}</b> Extra</span></div>'
        f'<div class="bar"><div style="background:{score_color}; width:{max(0, report.score):.0f}%;"></div></div>'
        f'{table}{untyped}{details}</div>'
    ))
    return report

# LLM judge for free-text fields whose fuzzy score is below the threshold.
# It is never consulted for numbers, enums, dates or ids, and every verdict is
# cached by (field, agent value, expected value), so re-running this cell is free.
from scripts.eval_lib import make_judge

judge = make_judge(
    lambda prompt: llm.chat(system="You are a strict grader. Answer with exactly one word: YES or NO.",
                            user=prompt, max_tokens=5, temperature=0.0),
    cache_path=f"{REPO}/.eval_judge_cache.json",
)
report = score_submission(REPO, PERSONA, agent_result, judge=judge)


## 10. 🏁 Wrap-Up

### ✅ What you built

An AI agent that autonomously files a Swiss tax return using a **Plan-then-Execute** architecture:

1. **Plan phase**: The LLM sees document *names*, not content, and decides what to read
2. **Selective retrieval**: Only the documents the plan requested are loaded
3. **Execute phase**: The LLM fills fields using only the relevant documents
4. **Visualization**: Every plan is rendered as a visual card showing the agent's reasoning

### 💡 Key insights

- **Selective retrieval matters**: Sending only relevant documents reduces noise and keeps context focused. This is especially important for smaller models with limited context windows.
- **Planning makes agents transparent**: By splitting reasoning into plan + execute, you can see *why* the agent made each decision — not just what it did.
- **The agent doesn't see the whole form at once**: It works page by page, building up context through `filled_history`.

### 🔭 Extensions to explore
- Add a **validation loop**: after filling a page, scan for validation errors and re-ask the LLM
- Use `ask_user` to handle ambiguous fields (the NPC taxpayer has verbal knowledge not in documents)
- Try different models on OpenRouter (change `OPENROUTER_MODEL`) and compare accuracy, speed and cost
- Build a **confidence score** into the plan — if the LLM is uncertain, flag it for human review

### 📚 References
- [OpenRouter Documentation](https://openrouter.ai/docs)
- [OpenAI Chat Completions API](https://platform.openai.com/docs/api-reference/chat)
- [Swiss Tax System Overview](https://www.estv.admin.ch/)
- [MCP Protocol Specification](https://modelcontextprotocol.io/)